# Building an Artificial Neural Network (ANN) using Keras

## Introduction

This notebook provides a comprehensive guide to building an Artificial Neural Network (ANN) using Keras. We'll cover all fundamental concepts with clear explanations.

### What is an Artificial Neural Network?

An **Artificial Neural Network (ANN)** is a computational model inspired by the human brain's biological neural networks. It consists of:

- **Neurons (Nodes)**: Basic processing units that receive inputs, process them, and produce outputs
- **Layers**: Collections of neurons organized in sequential layers
- **Weights**: Parameters that determine the strength of connections between neurons
- **Activation Functions**: Non-linear functions that help the network learn complex patterns

### Key Components:

1. **Input Layer**: Receives the input features
2. **Hidden Layers**: Intermediate layers that learn complex representations
3. **Output Layer**: Produces the final prediction

In this notebook, we'll build an ANN for a **binary classification problem** using the famous **Pima Indians Diabetes dataset**.

## Step 1: Import Required Libraries

We'll use:
- **NumPy**: For numerical computations
- **Pandas**: For data manipulation
- **Matplotlib/Seaborn**: For visualization
- **Scikit-learn**: For data preprocessing and evaluation
- **Keras**: High-level neural network API (part of TensorFlow)

In [ ]:
# Import essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import Keras and TensorFlow
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Import scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set random seeds for reproducibility
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)

# Configure plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

## Step 2: Load and Explore the Dataset

### About the Dataset:

The **Pima Indians Diabetes Dataset** contains medical diagnostic measurements to predict whether a patient has diabetes.

**Features:**
1. Pregnancies: Number of times pregnant
2. Glucose: Plasma glucose concentration
3. BloodPressure: Diastolic blood pressure (mm Hg)
4. SkinThickness: Triceps skin fold thickness (mm)
5. Insulin: 2-Hour serum insulin (mu U/ml)
6. BMI: Body mass index
7. DiabetesPedigreeFunction: Diabetes pedigree function
8. Age: Age in years

**Target:**
- Outcome: 0 (no diabetes) or 1 (diabetes)

In [ ]:
# Load the dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
column_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=column_names)

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Info:")
print(df.info())

print("\nStatistical Summary:")
print(df.describe())

print("\nTarget Distribution:")
print(df['Outcome'].value_counts())
print(f"\nClass Balance: {df['Outcome'].value_counts(normalize=True)}")

## Step 3: Data Preprocessing

### Why Preprocessing is Important:

1. **Feature Scaling**: Neural networks perform better when features are on similar scales
2. **Train-Test Split**: Prevents overfitting and allows proper evaluation
3. **Handling Missing Values**: Ensures data quality

### StandardScaler Explanation:

Standardization transforms features to have:
- **Mean = 0**
- **Standard Deviation = 1**

Formula: `z = (x - μ) / σ`

Where:
- x = original value
- μ = mean
- σ = standard deviation

In [ ]:
# Separate features and target
X = df.drop('Outcome', axis=1).values  # Features (8 columns)
y = df['Outcome'].values  # Target (binary: 0 or 1)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Split data into training and testing sets
# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTraining set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])

# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on training data
X_test_scaled = scaler.transform(X_test)  # Transform test data using training statistics

print("\nFeature scaling completed!")
print("Mean of scaled training data:", X_train_scaled.mean(axis=0).round(2))
print("Std of scaled training data:", X_train_scaled.std(axis=0).round(2))

## Step 4: Build the ANN Model Architecture

### Model Architecture Concepts:

#### 1. **Sequential Model**
A linear stack of layers where output from one layer feeds into the next.

#### 2. **Dense Layer (Fully Connected Layer)**
Every neuron in the layer is connected to every neuron in the previous layer.

Mathematical operation: `output = activation(dot(input, weights) + bias)`

#### 3. **Activation Functions**

- **ReLU (Rectified Linear Unit)**: `f(x) = max(0, x)`
  - Introduces non-linearity
  - Helps with vanishing gradient problem
  - Most commonly used for hidden layers

- **Sigmoid**: `f(x) = 1 / (1 + e^(-x))`
  - Outputs values between 0 and 1
  - Perfect for binary classification output

#### 4. **Dropout Layer**
Randomly sets a fraction of input units to 0 during training, which helps prevent overfitting.

#### 5. **Batch Normalization**
Normalizes the inputs of each layer, leading to faster training and better performance.

### Our Architecture:

```
Input (8 features)
    ↓
Dense(16) + ReLU + BatchNorm + Dropout(0.3)
    ↓
Dense(8) + ReLU + BatchNorm + Dropout(0.3)
    ↓
Dense(4) + ReLU + Dropout(0.2)
    ↓
Dense(1) + Sigmoid
    ↓
Output (probability: 0 to 1)
```

In [ ]:
# Create the Sequential model
model = Sequential(name='Diabetes_ANN')

# Input Layer + First Hidden Layer
# 16 neurons, ReLU activation
model.add(Dense(units=16, activation='relu', input_dim=8, name='input_layer'))
model.add(BatchNormalization())  # Normalize activations
model.add(Dropout(0.3))  # Drop 30% of neurons randomly

# Second Hidden Layer
# 8 neurons, ReLU activation
model.add(Dense(units=8, activation='relu', name='hidden_layer_1'))
model.add(BatchNormalization())
model.add(Dropout(0.3))

# Third Hidden Layer
# 4 neurons, ReLU activation
model.add(Dense(units=4, activation='relu', name='hidden_layer_2'))
model.add(Dropout(0.2))

# Output Layer
# 1 neuron with sigmoid activation for binary classification
model.add(Dense(units=1, activation='sigmoid', name='output_layer'))

# Display model architecture
print("Model Architecture:")
print("="*70)
model.summary()
print("="*70)

## Step 5: Compile the Model

### Compilation Concepts:

#### 1. **Optimizer: Adam**
Adaptive Moment Estimation - combines the best properties of AdaGrad and RMSProp optimizers.

**Key Features:**
- Adaptive learning rates for each parameter
- Computationally efficient
- Well-suited for large datasets

**Learning Rate**: Controls how much to change the model in response to error
- Too high: May overshoot optimal solution
- Too low: Training takes too long

#### 2. **Loss Function: Binary Crossentropy**
Measures the performance of a classification model whose output is a probability between 0 and 1.

Formula: `Loss = -[y*log(ŷ) + (1-y)*log(1-ŷ)]`

Where:
- y = actual label (0 or 1)
- ŷ = predicted probability

#### 3. **Metrics: Accuracy**
Percentage of correct predictions: `Accuracy = Correct Predictions / Total Predictions`

In [ ]:
# Define the optimizer with custom learning rate
optimizer = Adam(learning_rate=0.001)

# Compile the model
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',  # For binary classification
    metrics=['accuracy']  # Track accuracy during training
)

print("Model compiled successfully!")
print("\nOptimizer: Adam (learning_rate=0.001)")
print("Loss Function: Binary Crossentropy")
print("Metrics: Accuracy")

## Step 6: Define Callbacks

### Callback Concepts:

Callbacks are functions that can be applied at certain stages of the training process.

#### 1. **EarlyStopping**
Stops training when a monitored metric has stopped improving.

**Parameters:**
- `monitor`: Metric to watch (e.g., 'val_loss')
- `patience`: Number of epochs with no improvement after which training stops
- `restore_best_weights`: Restores model weights from the epoch with the best value

**Benefits:**
- Prevents overfitting
- Saves training time
- Automatically finds optimal training duration

#### 2. **ModelCheckpoint**
Saves the model after every epoch (or when improvement is detected).

**Parameters:**
- `filepath`: Path where to save the model
- `save_best_only`: Only save when the model is better than previous best
- `monitor`: Metric to monitor for improvement

In [ ]:
# Early Stopping: Stop training when validation loss doesn't improve
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=20,  # Wait 20 epochs before stopping
    restore_best_weights=True,  # Restore weights from best epoch
    verbose=1
)

# Model Checkpoint: Save the best model during training
checkpoint = ModelCheckpoint(
    filepath='best_diabetes_model.h5',
    monitor='val_accuracy',  # Monitor validation accuracy
    save_best_only=True,  # Only save if better than previous
    mode='max',  # Maximum accuracy is better
    verbose=1
)

print("Callbacks configured:")
print("1. EarlyStopping - Stops if val_loss doesn't improve for 20 epochs")
print("2. ModelCheckpoint - Saves best model based on val_accuracy")

## Step 7: Train the Model

### Training Concepts:

#### 1. **Epochs**
One complete pass through the entire training dataset.
- More epochs = more learning opportunities
- Too many epochs = overfitting risk

#### 2. **Batch Size**
Number of samples processed before updating model weights.

**Trade-offs:**
- Smaller batches: More frequent updates, more noise, slower
- Larger batches: Less noise, faster, may miss optimal solution

Common sizes: 16, 32, 64, 128

#### 3. **Validation Split**
Percentage of training data used for validation during training.
- Helps monitor overfitting
- Provides unbiased evaluation during training

#### 4. **Training Process:**

For each epoch:
1. **Forward Pass**: Input → Hidden Layers → Output
2. **Loss Calculation**: Compare predictions with actual labels
3. **Backward Pass**: Calculate gradients (how much each weight contributed to error)
4. **Weight Update**: Adjust weights using optimizer
5. **Validation**: Evaluate on validation set (no weight updates)

In [ ]:
# Train the model
print("Starting model training...\n")

history = model.fit(
    X_train_scaled,  # Scaled training features
    y_train,  # Training labels
    epochs=150,  # Maximum number of epochs
    batch_size=32,  # Process 32 samples at a time
    validation_split=0.2,  # Use 20% of training data for validation
    callbacks=[early_stopping, checkpoint],  # Apply callbacks
    verbose=1  # Show progress bar
)

print("\n" + "="*70)
print("Training completed!")
print("="*70)

## Step 8: Visualize Training History

### Understanding Training Curves:

#### 1. **Loss Curves**
- **Decreasing loss**: Model is learning
- **Training loss << Validation loss**: Overfitting
- **Both losses high**: Underfitting
- **Both losses decreasing together**: Good learning

#### 2. **Accuracy Curves**
- **Increasing accuracy**: Model improving
- **Training accuracy >> Validation accuracy**: Overfitting
- **Both accuracies plateauing**: Model has converged

#### 3. **Ideal Scenario:**
- Training and validation curves move together
- Minimal gap between them
- Smooth improvement over epochs

In [ ]:
# Extract training history
train_loss = history.history['loss']
val_loss = history.history['val_loss']
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
epochs_range = range(1, len(train_loss) + 1)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot Loss
axes[0].plot(epochs_range, train_loss, 'b-', label='Training Loss', linewidth=2)
axes[0].plot(epochs_range, val_loss, 'r-', label='Validation Loss', linewidth=2)
axes[0].set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (Binary Crossentropy)', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot Accuracy
axes[1].plot(epochs_range, train_acc, 'b-', label='Training Accuracy', linewidth=2)
axes[1].plot(epochs_range, val_acc, 'r-', label='Validation Accuracy', linewidth=2)
axes[1].set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Total epochs trained: {len(train_loss)}")
print(f"Final training loss: {train_loss[-1]:.4f}")
print(f"Final validation loss: {val_loss[-1]:.4f}")
print(f"Final training accuracy: {train_acc[-1]:.4f}")
print(f"Final validation accuracy: {val_acc[-1]:.4f}")

## Step 9: Evaluate on Test Data

### Evaluation Metrics:

#### 1. **Confusion Matrix**
A table showing actual vs predicted classifications:

```
                Predicted
              No    Yes
Actual  No   TN    FP
        Yes  FN    TP
```

- **TN (True Negative)**: Correctly predicted no diabetes
- **TP (True Positive)**: Correctly predicted diabetes
- **FN (False Negative)**: Predicted no diabetes, but has diabetes (Type II error)
- **FP (False Positive)**: Predicted diabetes, but doesn't have it (Type I error)

#### 2. **Classification Metrics:**

- **Precision**: TP / (TP + FP)
  - Of all predicted positives, how many are actually positive?
  - Important when false positives are costly

- **Recall (Sensitivity)**: TP / (TP + FN)
  - Of all actual positives, how many did we catch?
  - Important when false negatives are costly

- **F1-Score**: 2 * (Precision * Recall) / (Precision + Recall)
  - Harmonic mean of precision and recall
  - Balances both metrics

- **Accuracy**: (TP + TN) / (TP + TN + FP + FN)
  - Overall correctness

In [ ]:
# Evaluate on test set
print("Evaluating model on test data...\n")
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

print("="*70)
print("TEST SET RESULTS")
print("="*70)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print("="*70)

## Step 10: Make Predictions and Analyze Results

In [ ]:
# Make predictions on test set
y_pred_prob = model.predict(X_test_scaled)  # Get probabilities
y_pred = (y_pred_prob > 0.5).astype(int).flatten()  # Convert to binary (0 or 1)

print("Sample predictions:")
print("Actual | Predicted | Probability")
print("-" * 40)
for i in range(10):
    print(f"  {y_test[i]}    |     {y_pred[i]}     |   {y_pred_prob[i][0]:.4f}")

## Step 11: Confusion Matrix Visualization

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['No Diabetes (0)', 'Diabetes (1)'],
            yticklabels=['No Diabetes (0)', 'Diabetes (1)'],
            annot_kws={'size': 16, 'weight': 'bold'})
plt.title('Confusion Matrix - Diabetes Prediction', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Actual', fontsize=13, fontweight='bold')
plt.xlabel('Predicted', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Extract values
tn, fp, fn, tp = cm.ravel()
print("\nConfusion Matrix Breakdown:")
print(f"True Negatives (TN): {tn} - Correctly predicted no diabetes")
print(f"False Positives (FP): {fp} - Incorrectly predicted diabetes")
print(f"False Negatives (FN): {fn} - Missed diabetes cases")
print(f"True Positives (TP): {tp} - Correctly predicted diabetes")

## Step 12: Detailed Classification Report

In [ ]:
# Generate classification report
print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, y_pred, 
                           target_names=['No Diabetes', 'Diabetes'],
                           digits=4))
print("="*70)

# Calculate additional metrics
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\nKey Metrics Summary:")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {precision:.4f} (Of predicted diabetes cases, {precision*100:.2f}% were correct)")
print(f"Recall:    {recall:.4f} (Caught {recall*100:.2f}% of actual diabetes cases)")
print(f"F1-Score:  {f1:.4f} (Harmonic mean of precision and recall)")

## Step 13: Making Predictions on New Data

### How to Use the Model for Prediction:

1. Prepare new data in the same format as training data
2. Apply the same scaling transformation
3. Use `model.predict()` to get probability
4. Apply threshold (0.5) to convert to binary prediction

In [ ]:
# Example: Predict for a new patient
new_patient = np.array([[6, 148, 72, 35, 0, 33.6, 0.627, 50]])  # Example data

print("New Patient Data:")
print(f"Pregnancies: {new_patient[0][0]}")
print(f"Glucose: {new_patient[0][1]}")
print(f"Blood Pressure: {new_patient[0][2]}")
print(f"Skin Thickness: {new_patient[0][3]}")
print(f"Insulin: {new_patient[0][4]}")
print(f"BMI: {new_patient[0][5]}")
print(f"Diabetes Pedigree Function: {new_patient[0][6]}")
print(f"Age: {new_patient[0][7]}")

# Scale the new data using the same scaler
new_patient_scaled = scaler.transform(new_patient)

# Make prediction
prediction_prob = model.predict(new_patient_scaled)[0][0]
prediction_class = int(prediction_prob > 0.5)

print("\n" + "="*70)
print("PREDICTION RESULT")
print("="*70)
print(f"Probability of Diabetes: {prediction_prob:.4f} ({prediction_prob*100:.2f}%)")
print(f"Prediction: {'DIABETES DETECTED' if prediction_class == 1 else 'NO DIABETES'}")
print("="*70)

## Step 14: Save and Load the Model

### Model Persistence:

You can save trained models for later use:

1. **Save entire model**: Architecture + weights + optimizer state
2. **Save only weights**: Just the learned parameters
3. **SavedModel format**: TensorFlow's recommended format

In [ ]:
# Save the complete model
model.save('diabetes_ann_model.h5')
print("Model saved as 'diabetes_ann_model.h5'")

# To load the model later:
# loaded_model = keras.models.load_model('diabetes_ann_model.h5')
# predictions = loaded_model.predict(new_data_scaled)

print("\nTo load this model in the future, use:")
print("loaded_model = keras.models.load_model('diabetes_ann_model.h5')")

## Summary and Key Takeaways

### What We Learned:

1. **Data Preprocessing**:
   - Feature scaling is crucial for neural networks
   - Train-test split prevents overfitting
   - StandardScaler normalizes features

2. **Model Architecture**:
   - Sequential model for linear stack of layers
   - Dense layers for fully connected networks
   - Dropout and BatchNormalization for regularization
   - ReLU for hidden layers, Sigmoid for binary output

3. **Training**:
   - Adam optimizer adapts learning rates
   - Binary crossentropy for binary classification
   - Callbacks (EarlyStopping, ModelCheckpoint) improve training
   - Batch size and epochs are hyperparameters to tune

4. **Evaluation**:
   - Confusion matrix shows prediction breakdown
   - Precision, Recall, F1-score provide detailed insights
   - Training curves help diagnose overfitting/underfitting

### Next Steps to Improve:

1. **Hyperparameter Tuning**:
   - Try different network architectures
   - Adjust learning rate
   - Experiment with dropout rates
   - Change batch size and epochs

2. **Advanced Techniques**:
   - Cross-validation for robust evaluation
   - Learning rate scheduling
   - Data augmentation (if applicable)
   - Ensemble methods

3. **Feature Engineering**:
   - Create new features from existing ones
   - Handle missing values more carefully
   - Remove outliers

### Resources for Further Learning:

- Keras Documentation: https://keras.io/
- TensorFlow Tutorials: https://www.tensorflow.org/tutorials
- Deep Learning Book by Ian Goodfellow
- Neural Networks and Deep Learning by Michael Nielsen